In [1]:
# model_training.ipynb
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import librosa
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuration
BALANCED_PATH = '../data/balanced/mitral_valve'
OUTPUT_PATH = '../exported_models/mitral_classifier'
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, 'checkpoints'), exist_ok=True)

In [ ]:


class HeartSoundDataGenerator:
    """Custom data generator for heart sound spectrograms"""
    
    def __init__(self, data_dir, df, batch_size=32, target_size=(224, 224), 
                 use_augmentation=False):
        self.data_dir = data_dir
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.target_size = target_size
        self.use_augmentation = use_augmentation
        self.classes = ['normal', 'rhd']
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        # Parameters for mel spectrogram
        self.n_mels = 128
        self.n_fft = 2048
        self.hop_length = 512
        
    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))
    
    def __getitem__(self, idx):
        batch_df = self.df[idx * self.batch_size:(idx + 1) * self.batch_size]
        
        X_batch = []
        y_batch = []
        
        for _, row in batch_df.iterrows():
            patient_id = row['patient_id']
            label = row['label']
            
            # Try different paths
            file_path = None
            
            # Check in main balanced directory
            main_path = os.path.join(self.data_dir, label, f"{patient_id}.wav")
            if os.path.exists(main_path):
                file_path = main_path
            else:
                # Check in augmented directory
                aug_path = os.path.join(self.data_dir, 'augmented', label, f"{patient_id}.wav")
                if os.path.exists(aug_path):
                    file_path = aug_path
                else:
                    # Try to find any file starting with patient_id
                    aug_dir = os.path.join(self.data_dir, 'augmented', label)
                    if os.path.exists(aug_dir):
                        for f in os.listdir(aug_dir):
                            if f.startswith(str(patient_id)):
                                file_path = os.path.join(aug_dir, f)
                                break
            
            if file_path is None:
                print(f"Warning: File not found for {patient_id}")
                placeholder = np.zeros((*self.target_size, 3))
                X_batch.append(placeholder)
                y_batch.append(0)
                continue
            
            try:
                # Load audio (already bandpass filtered from preprocessing)
                signal, fs = librosa.load(file_path, sr=4000)
                
                # Generate mel spectrogram
                mel_spec = librosa.feature.melspectrogram(
                    y=signal, 
                    sr=fs,
                    n_mels=self.n_mels,
                    n_fft=self.n_fft,
                    hop_length=self.hop_length,
                    fmin=20,    # Keep within bandpass range
                    fmax=400
                )
                
                # Log compression
                mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
                
                # Normalize to [0, 1]
                mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min() + 1e-8)
                
                # Resize to target size
                mel_spec = tf.image.resize(
                    mel_spec[..., np.newaxis], 
                    self.target_size
                ).numpy()
                
                # Convert to 3-channel for MobileNetV2
                mel_spec = np.repeat(mel_spec, 3, axis=-1)
                
                X_batch.append(mel_spec)
                y_batch.append(self.class_to_idx[label])
                
            except Exception as e:
                print(f"Error processing {file_path}: {e}")
                placeholder = np.zeros((*self.target_size, 3))
                X_batch.append(placeholder)
                y_batch.append(0)
        
        # Preprocess for MobileNetV2
        X_batch = np.array(X_batch)
        X_batch = preprocess_input(X_batch)
        y_batch = tf.keras.utils.to_categorical(y_batch, num_classes=2)
        
        return X_batch, y_batch

def create_mobilenetv2_model(input_shape=(224, 224, 3), num_classes=2):
    """Create MobileNetV2 model for heart sound classification"""
    
    # Base model
    base_model = MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False
    
    # Custom head with regularization
    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs=base_model.input, outputs=outputs)
    
    return model

def create_data_splits():
    """Create train/val/test splits from combined data"""
    
    # Load combined data
    combined_df = pd.read_csv(os.path.join(BALANCED_PATH, 'combined_summary.csv'))
    
    # Separate original and augmented
    original_df = combined_df[~combined_df['patient_id'].astype(str).str.contains('_', na=False)].copy()
    augmented_df = combined_df[combined_df['patient_id'].astype(str).str.contains('_', na=False)].copy()
    
    print(f"Original samples: {len(original_df)}")
    print(f"Augmented samples: {len(augmented_df)}")
    
    # Split original patients stratified by label
    patients = original_df['patient_id'].unique()
    labels = []
    for patient in patients:
        label = original_df[original_df['patient_id'] == patient]['label'].iloc[0]
        labels.append(label)
    
    # First split: 70% train, 30% test
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
    train_idx, test_idx = next(sss.split(patients, labels))
    train_patients = patients[train_idx]
    test_patients = patients[test_idx]
    
    # Split test into validation and test (50/50)
    test_labels = [labels[i] for i in test_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
    val_idx, test_final_idx = next(sss2.split(test_patients, test_labels))
    val_patients = test_patients[val_idx]
    test_patients = test_patients[test_final_idx]
    
    # Create DataFrames
    train_df = original_df[original_df['patient_id'].isin(train_patients)]
    val_df = original_df[original_df['patient_id'].isin(val_patients)]
    test_df = original_df[original_df['patient_id'].isin(test_patients)]
    
    # Add augmented data to training
    train_df = pd.concat([train_df, augmented_df], ignore_index=True)
    
    # Shuffle training data
    train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"\nTrain samples: {len(train_df)}")
    print(f"Val samples: {len(val_df)}")
    print(f"Test samples: {len(test_df)}")
    
    print("\nTrain distribution:")
    print(train_df['label'].value_counts())
    print("\nVal distribution:")
    print(val_df['label'].value_counts())
    print("\nTest distribution:")
    print(test_df['label'].value_counts())
    
    return train_df, val_df, test_df

def train_model(model, train_gen, val_gen, epochs=50):
    """Train the model with class weights"""
    
    # Calculate class weights for imbalance
    train_labels = train_gen.df['label'].values
    class_counts = np.bincount([train_gen.class_to_idx[l] for l in train_labels])
    total = len(train_labels)
    class_weights = {i: total / (len(class_counts) * count) for i, count in enumerate(class_counts)}
    print(f"Class weights: {class_weights}")
    
    # Callbacks
    checkpoint = callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_PATH, 'checkpoints', 'best_model.h5'),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
    
    early_stopping = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    )
    
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
    
    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    
    # Train
    history = model.fit(
        train_gen,
        epochs=epochs,
        validation_data=val_gen,
        class_weight=class_weights,
        callbacks=[checkpoint, early_stopping, reduce_lr],
        verbose=1
    )
    
    return history

def plot_training_history(history):
    """Plot training history"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Accuracy
    axes[0, 0].plot(history.history['accuracy'], label='Train', linewidth=2)
    axes[0, 0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
    axes[0, 0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Loss
    axes[0, 1].plot(history.history['loss'], label='Train', linewidth=2)
    axes[0, 1].plot(history.history['val_loss'], label='Validation', linewidth=2)
    axes[0, 1].set_title('Model Loss', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # AUC
    if 'auc' in history.history:
        axes[1, 0].plot(history.history['auc'], label='Train', linewidth=2)
        axes[1, 0].plot(history.history['val_auc'], label='Validation', linewidth=2)
        axes[1, 0].set_title('Model AUC', fontsize=14, fontweight='bold')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('AUC')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
    
    # Learning rate
    if 'lr' in history.history:
        axes[1, 1].plot(history.history['lr'], linewidth=2)
        axes[1, 1].set_title('Learning Rate', fontsize=14, fontweight='bold')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('LR')
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'training_history.png'), dpi=150, bbox_inches='tight')
    plt.show()

def evaluate_model(model, test_gen, test_df):
    """Evaluate model on test set"""
    
    # Get predictions
    y_true = []
    y_pred = []
    patient_ids = []
    
    for batch in range(len(test_gen)):
        X_batch, y_batch = test_gen[batch]
        pred_batch = model.predict(X_batch, verbose=0)
        
        start_idx = batch * test_gen.batch_size
        end_idx = min(start_idx + test_gen.batch_size, len(test_gen.df))
        batch_patients = test_gen.df.iloc[start_idx:end_idx]['patient_id'].values
        
        y_true.extend(np.argmax(y_batch, axis=1))
        y_pred.extend(pred_batch[:, 1])  # Probability for class 1 (RHD)
        patient_ids.extend(batch_patients)
    
    y_true = np.array(y_true)
    y_pred_proba = np.array(y_pred)
    y_pred_class = (y_pred_proba > 0.5).astype(int)
    
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred_class)
    precision = precision_score(y_true, y_pred_class, average='binary')
    recall = recall_score(y_true, y_pred_class, average='binary')
    f1 = f1_score(y_true, y_pred_class, average='binary')
    auc = roc_auc_score(y_true, y_pred_proba)
    
    print("\n" + "="*50)
    print("TEST RESULTS")
    print("="*50)
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC:       {auc:.4f}")
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred_class, 
                               target_names=['Normal', 'RHD']))
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred_class)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Confusion matrix as heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Normal', 'RHD'],
                yticklabels=['Normal', 'RHD'])
    axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('True Label')
    axes[0].set_xlabel('Predicted Label')
    
    # ROC curve
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    axes[1].plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.3f})', linewidth=2)
    axes[1].plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'evaluation_results.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    # Create results DataFrame
    results_df = pd.DataFrame({
        'patient_id': patient_ids[:len(y_true)],
        'true_label': y_true,
        'pred_proba_rhd': y_pred_proba,
        'pred_label': y_pred_class
    })
    results_df['true_label_name'] = results_df['true_label'].map({0: 'Normal', 1: 'RHD'})
    results_df['pred_label_name'] = results_df['pred_label'].map({0: 'Normal', 1: 'RHD'})
    results_df['correct'] = results_df['true_label'] == results_df['pred_label']
    
    # Show misclassified examples
    misclassified = results_df[~results_df['correct']]
    if len(misclassified) > 0:
        print(f"\nMisclassified samples: {len(misclassified)}")
        print(misclassified[['patient_id', 'true_label_name', 'pred_label_name', 'pred_proba_rhd']].head(10))
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc,
        'confusion_matrix': cm,
        'results_df': results_df
    }

def fine_tune_model(model, train_gen, val_gen, epochs=20):
    """Fine-tune the model by unfreezing some layers"""
    
    # Unfreeze top layers
    model.trainable = True
    
    # Freeze early layers (keep first 100 layers frozen)
    for layer in model.layers[:100]:
        layer.trainable = False
    
    # Recompile with lower learning rate
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    
    # Callbacks
    checkpoint = callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_PATH, 'checkpoints', 'finetuned_model.h5'),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
    
    early_stopping = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
    
    # Fine-tune
    history = model.fit(
        train_gen,
        epochs=epochs,
        validation_data=val_gen,
        callbacks=[checkpoint, early_stopping],
        verbose=1
    )
    
    return history

if __name__ == "__main__":
    print("="*60)
    print("MITRAL VALVE MURMUR CLASSIFICATION WITH MOBILENETV2")
    print("="*60)
    
    # Create data splits
    print("\n1. Creating data splits...")
    train_df, val_df, test_df = create_data_splits()
    
    # Create data generators
    batch_size = 16
    print(f"\n2. Creating data generators (batch_size={batch_size})...")
    
    train_gen = HeartSoundDataGenerator(
        BALANCED_PATH, train_df, batch_size=batch_size,
        use_augmentation=True
    )
    
    val_gen = HeartSoundDataGenerator(
        BALANCED_PATH, val_df, batch_size=batch_size,
        use_augmentation=False
    )
    
    test_gen = HeartSoundDataGenerator(
        BALANCED_PATH, test_df, batch_size=batch_size,
        use_augmentation=False
    )
    
    # Create model
    print("\n3. Building MobileNetV2 model...")
    model = create_mobilenetv2_model()
    model.summary()
    
    # Train model
    print("\n4. Training model...")
    history = train_model(model, train_gen, val_gen, epochs=50)
    
    # Plot training history
    print("\n5. Plotting training history...")
    plot_training_history(history)
    
    # Evaluate on test set
    print("\n6. Evaluating on test set...")
    metrics = evaluate_model(model, test_gen, test_df)
    
    # Fine-tune model
    print("\n7. Fine-tuning model...")
    fine_tune_history = fine_tune_model(model, train_gen, val_gen, epochs=20)
    
    # Evaluate fine-tuned model
    print("\n8. Evaluating fine-tuned model on test set...")
    fine_tuned_metrics = evaluate_model(model, test_gen, test_df)
    
    # Save final model
    model.save(os.path.join(OUTPUT_PATH, 'final_model.h5'))
    print(f"\n✅ Model saved to {OUTPUT_PATH}/final_model.h5")
    
    # Save metrics
    metrics_df = pd.DataFrame({
        'Phase': ['Initial', 'Fine-tuned'],
        'Accuracy': [metrics['accuracy'], fine_tuned_metrics['accuracy']],
        'Precision': [metrics['precision'], fine_tuned_metrics['precision']],
        'Recall': [metrics['recall'], fine_tuned_metrics['recall']],
        'F1': [metrics['f1'], fine_tuned_metrics['f1']],
        'AUC': [metrics['auc'], fine_tuned_metrics['auc']]
    })
    metrics_df.to_csv(os.path.join(OUTPUT_PATH, 'model_metrics.csv'), index=False)
    print("\n✅ Metrics saved to model_metrics.csv")
    
    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print("="*60)